# Minkowski Tensor Additivity by `center` Mode

Minkowski tensors satisfy an **additivity property**: for two disjoint bodies $A$ and $B$,

$$W(A \cup B) = W(A) + W(B)$$

This holds unconditionally for all tensors when `center=None` (origin as reference).
When a non-trivial `center` mode is used, the reference frame shifts before computing
position-dependent quantities, which can break additivity.

This notebook tests additivity in two ways:

- **Part 1 — No labels**: compute $W(A)$ and $W(B)$ on individual sub-meshes, compare sum to the combined mesh $W(A \cup B)$.
- **Part 2 — With labels**: pass both bodies as one mesh with a per-face label array; only the `centroid_mesh` cases are tested since the other modes add no new information in the labelled setting.

| `center` mode | `center_per_label` | Scalars | Vectors | Rank-2 tensors |
|---|---|---|---|---|
| `None` | — | ✓ | ✓ | ✓ |
| `None` + manual global centroid pre-shift | — | ✓ | ✓ | ✓ |
| `'centroid_mesh'` | `True` | ✓ | ✗ | ✗ |
| `'centroid_mesh'` | `False` | ✓ | ✓ | ✓ |
| `'reference_centroid'` | — | ✓ | ✓ | ✗ |

*(Cells below confirm this table numerically.)*


In [1]:
import numpy as np
import pykarambola
from pykarambola import minkowski_tensors

print(f"pykarambola version: {pykarambola.__version__}")

pykarambola version: 0.2.0


## Helper functions

In [2]:
def box_mesh(lx=1.0, ly=1.0, lz=1.0, center=(0.0, 0.0, 0.0)):
    """Triangulated axis-aligned box with outward-facing normals.

    Parameters
    ----------
    lx, ly, lz : float
        Side lengths.
    center : (3,) array_like
        Position of the box centre.

    Returns
    -------
    verts : (8, 3) ndarray
    faces : (12, 3) ndarray
    """
    cx, cy, cz = center
    ha, hb, hc = lx / 2.0, ly / 2.0, lz / 2.0
    verts = np.array([
        [cx - ha, cy - hb, cz - hc],  # 0
        [cx + ha, cy - hb, cz - hc],  # 1
        [cx + ha, cy + hb, cz - hc],  # 2
        [cx - ha, cy + hb, cz - hc],  # 3
        [cx - ha, cy - hb, cz + hc],  # 4
        [cx + ha, cy - hb, cz + hc],  # 5
        [cx + ha, cy + hb, cz + hc],  # 6
        [cx - ha, cy + hb, cz + hc],  # 7
    ], dtype=np.float64)
    faces = np.array([
        [0, 3, 2], [0, 2, 1],  # -z face
        [4, 5, 6], [4, 6, 7],  # +z face
        [0, 1, 5], [0, 5, 4],  # -y face
        [2, 3, 7], [2, 7, 6],  # +y face
        [0, 4, 7], [0, 7, 3],  # -x face
        [1, 2, 6], [1, 6, 5],  # +x face
    ], dtype=np.int64)
    return verts, faces


SCALAR_KEYS = ['w000', 'w100', 'w200', 'w300']
VECTOR_KEYS = ['w010', 'w110', 'w210', 'w310']
TENSOR_KEYS = ['w020', 'w120', 'w220', 'w320', 'w102', 'w202']


def additivity_table(wA, wB, wAB, tol=1e-8):
    """Print whether W(A) + W(B) = W(A cup B) for each functional.

    Parameters
    ----------
    wA, wB : dict
        Results for body A and body B.
    wAB : dict
        Result for the combined mesh.
    tol : float
        Absolute tolerance for the additivity check.
    """
    header = f"  {'Key':<8}  {'Additive?':<12}  max|W(A cup B) - W(A) - W(B)|"
    print(header)
    print('  ' + '-' * (len(header) - 2))
    for category, keys in [('Scalars', SCALAR_KEYS),
                            ('Vectors', VECTOR_KEYS),
                            ('Rank-2 tensors', TENSOR_KEYS)]:
        print(f"  [{category}]")
        for key in keys:
            if key not in wA:
                continue
            a  = np.asarray(wA[key])
            b  = np.asarray(wB[key])
            ab = np.asarray(wAB[key])
            diff = float(np.max(np.abs(ab - (a + b))))
            mark = "yes" if diff < tol else "NO"
            print(f"    {key:<8}  {mark:<12}  {diff:.3e}")


def two_box_mesh(vA, fA, vB, fB):
    """Combine two sub-meshes into one mesh with per-face labels."""
    fB_global = fB + len(vA)
    verts  = np.vstack([vA, vB])
    faces  = np.vstack([fA, fB_global])
    labels = np.array([1] * len(fA) + [2] * len(fB), dtype=np.int64)
    return verts, faces, labels

---
# Asymmetric pair — two boxes of different sizes

Two boxes of **different sizes**:
- **Box A** (label 1): 2×2×2 (volume = 8) at `(0, 0, 0)`
- **Box B** (label 2): 4×4×4 (volume = 64) at `(20, 0, 0)`

The global volume-weighted centroid is no longer equidistant from the two boxes:

$$c_{\text{global}} = \frac{8 \cdot 0 + 64 \cdot 20}{8 + 64} \approx (17.78,\, 0,\, 0)$$

After this shift, Box A sits at $(-17.78, 0, 0)$ and Box B at $(2.22, 0, 0)$.
The surface-area moments of the two boxes no longer cancel, so vectors are
**not** additive for `centroid_mesh` with a per-body centroid shift.


In [3]:
vA_asy, fA_asy = box_mesh(lx=2.0, ly=2.0, lz=2.0, center=( 0.0, 0.0, 0.0))  # small
vB_asy, fB_asy = box_mesh(lx=4.0, ly=4.0, lz=4.0, center=(20.0, 0.0, 0.0))  # large
verts_asy, faces_asy, labels_asy = two_box_mesh(vA_asy, fA_asy, vB_asy, fB_asy)

wAB_asy_none = minkowski_tensors(verts_asy, faces_asy, center=None)
com_asy = np.array(wAB_asy_none['w010']) / wAB_asy_none['w000']

print(f"Box A : {len(vA_asy):2d} verts, {len(fA_asy):2d} faces  (2x2x2, vol=8,  label 1, centre at  (0, 0, 0))")
print(f"Box B : {len(vB_asy):2d} verts, {len(fB_asy):2d} faces  (4x4x4, vol=64, label 2, centre at (20, 0, 0))")
print(f"Total : {len(verts_asy):2d} verts, {len(faces_asy):2d} faces")
print(f"Global centroid: {np.round(com_asy, 4)}")

Box A :  8 verts, 12 faces  (2x2x2, vol=8,  label 1, centre at  (0, 0, 0))
Box B :  8 verts, 12 faces  (4x4x4, vol=64, label 2, centre at (20, 0, 0))
Total : 16 verts, 24 faces
Global centroid: [17.7778  0.      0.    ]


## Part 1: No labels


In [4]:
wA_asy  = minkowski_tensors(vA_asy,    fA_asy,    center=None)
wB_asy  = minkowski_tensors(vB_asy,    fB_asy,    center=None)
# wAB_asy_none already computed above

print("center=None  (asymmetric)")
additivity_table(wA_asy, wB_asy, wAB_asy_none)

center=None  (asymmetric)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           2.842e-14
    w100      yes           0.000e+00
    w200      yes           3.553e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           0.000e+00
    w110      yes           0.000e+00
    w210      yes           0.000e+00
    w310      yes           1.110e-16
  [Rank-2 tensors]
    w020      yes           0.000e+00
    w120      yes           1.421e-14
    w220      yes           9.095e-13
    w320      yes           6.647e-15
    w102      yes           0.000e+00
    w202      yes           8.882e-16


Note in the case below the same global centroid is subtracted from all the cases, the individual and the multibox mesh. 

In [5]:
vA_asy_s    = vA_asy    - com_asy
vB_asy_s    = vB_asy    - com_asy
verts_asy_s = verts_asy - com_asy

wA_asy_s  = minkowski_tensors(vA_asy_s,    fA_asy,    center=None)
wB_asy_s  = minkowski_tensors(vB_asy_s,    fB_asy,    center=None)
wAB_asy_s = minkowski_tensors(verts_asy_s, faces_asy, center=None)

print("center=None + manual global centroid pre-shift  (asymmetric)")
additivity_table(wA_asy_s, wB_asy_s, wAB_asy_s)

center=None + manual global centroid pre-shift  (asymmetric)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           0.000e+00
    w100      yes           0.000e+00
    w200      yes           3.553e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           2.132e-14
    w110      yes           1.421e-14
    w210      yes           1.421e-14
    w310      yes           1.421e-14
  [Rank-2 tensors]
    w020      yes           0.000e+00
    w120      yes           9.095e-13
    w220      yes           4.547e-13
    w320      yes           3.553e-15
    w102      yes           0.000e+00
    w202      yes           8.882e-16


In this case each individual mesh is moved by its respective COM, as per current pykarambola iplementation. 

In [6]:
wA_asy_cm  = minkowski_tensors(vA_asy,    fA_asy,    center='centroid_mesh')
wB_asy_cm  = minkowski_tensors(vB_asy,    fB_asy,    center='centroid_mesh')
wAB_asy_cm = minkowski_tensors(verts_asy, faces_asy, center='centroid_mesh')

print("center='centroid_mesh'  (asymmetric — vectors now non-additive)")
additivity_table(wA_asy_cm, wB_asy_cm, wAB_asy_cm)

center='centroid_mesh'  (asymmetric — vectors now non-additive)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           0.000e+00
    w100      yes           0.000e+00
    w200      yes           3.553e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           7.816e-14
    w110      NO            7.111e+01
    w210      NO            8.378e+01
    w310      NO            6.516e+01
  [Rank-2 tensors]
    w020      NO            2.844e+03
    w120      NO            2.686e+03
    w220      NO            2.048e+03
    w320      NO            1.345e+03
    w102      yes           0.000e+00
    w202      yes           8.882e-16


In [7]:
wA_asy_rc  = minkowski_tensors(vA_asy,    fA_asy,    center='reference_centroid')
wB_asy_rc  = minkowski_tensors(vB_asy,    fB_asy,    center='reference_centroid')
wAB_asy_rc = minkowski_tensors(verts_asy, faces_asy, center='reference_centroid')

print("center='reference_centroid'  (asymmetric)")
additivity_table(wA_asy_rc, wB_asy_rc, wAB_asy_rc)

center='reference_centroid'  (asymmetric)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           2.842e-14
    w100      yes           0.000e+00
    w200      yes           3.553e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           0.000e+00
    w110      yes           0.000e+00
    w210      yes           0.000e+00
    w310      yes           1.110e-16
  [Rank-2 tensors]
    w020      NO            2.844e+03
    w120      NO            2.560e+03
    w220      NO            1.676e+03
    w320      NO            8.378e+02
    w102      yes           0.000e+00
    w202      yes           8.882e-16


## Part 2: With labels
When laebles are passed, the default option is center_per_label=True


In [ ]:
res_asy_cm_pl = minkowski_tensors(
    verts_asy, faces_asy, labels=labels_asy,
    center='centroid_mesh', center_per_label=True,
)
print("center='centroid_mesh', center_per_label=True  (with labels, asymmetric)")
additivity_table(res_asy_cm_pl[1], res_asy_cm_pl[2], wAB_asy_cm)

center='centroid_mesh', center_per_label=True  (with labels, asymmetric)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           0.000e+00
    w100      yes           0.000e+00
    w200      yes           3.553e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           7.816e-14
    w110      NO            7.111e+01
    w210      NO            8.378e+01
    w310      NO            6.516e+01
  [Rank-2 tensors]
    w020      NO            2.844e+03
    w120      NO            2.686e+03
    w220      NO            2.048e+03
    w320      NO            1.345e+03
    w102      yes           0.000e+00
    w202      yes           8.882e-16


In [9]:
res_asy_cm_gl = minkowski_tensors(
    verts_asy, faces_asy, labels=labels_asy,
    center='centroid_mesh', center_per_label=False,
)
print("center='centroid_mesh', center_per_label=False  (with labels, asymmetric)")
additivity_table(res_asy_cm_gl[1], res_asy_cm_gl[2], wAB_asy_cm)

center='centroid_mesh', center_per_label=False  (with labels, asymmetric)
  Key       Additive?     max|W(A cup B) - W(A) - W(B)|
  -----------------------------------------------------
  [Scalars]
    w000      yes           0.000e+00
    w100      yes           0.000e+00
    w200      yes           3.553e-15
    w300      yes           0.000e+00
  [Vectors]
    w010      yes           2.132e-14
    w110      yes           1.421e-14
    w210      yes           1.421e-14
    w310      yes           1.421e-14
  [Rank-2 tensors]
    w020      yes           0.000e+00
    w120      yes           4.547e-13
    w220      yes           4.547e-13
    w320      yes           3.664e-15
    w102      yes           0.000e+00
    w202      yes           8.882e-16


---
## Summary


In [10]:
TOL = 1e-8

def cat_ok(wA, wB, wAB, keys):
    diffs = [float(np.max(np.abs(
        np.asarray(wAB[k]) - np.asarray(wA[k]) - np.asarray(wB[k])
    ))) for k in keys if k in wA]
    return "yes" if all(d < TOL for d in diffs) else "NO"

rows = [
    # --- no labels ---
    ("[no lbl]  center=None",                         wA_asy,           wB_asy,           wAB_asy_none),
    ("[no lbl]  center=None + manual pre-shift",      wA_asy_s,         wB_asy_s,         wAB_asy_s   ),
    ("[no lbl]  centroid_mesh",                       wA_asy_cm,        wB_asy_cm,        wAB_asy_cm  ),
    ("[no lbl]  reference_centroid",                  wA_asy_rc,        wB_asy_rc,        wAB_asy_rc  ),
    # --- with labels ---
    ("[lbl]     centroid_mesh, center_per_label=True",  res_asy_cm_pl[1], res_asy_cm_pl[2], wAB_asy_cm),
    ("[lbl]     centroid_mesh, center_per_label=False", res_asy_cm_gl[1], res_asy_cm_gl[2], wAB_asy_cm),
]

print(f"{'Mode':<52}  {'Scalars':<9}  {'Vectors':<9}  {'Rank-2'}")
print("-" * 85)
prev_prefix = None
for desc, a, b, ab in rows:
    prefix = desc[:6]
    if prev_prefix and prefix != prev_prefix:
        print()
    prev_prefix = prefix
    print(f"  {desc:<50}  {cat_ok(a,b,ab,SCALAR_KEYS):<9}  "
          f"{cat_ok(a,b,ab,VECTOR_KEYS):<9}  {cat_ok(a,b,ab,TENSOR_KEYS)}")


Mode                                                  Scalars    Vectors    Rank-2
-------------------------------------------------------------------------------------
  [no lbl]  center=None                               yes        yes        yes
  [no lbl]  center=None + manual pre-shift            yes        yes        yes
  [no lbl]  centroid_mesh                             yes        NO         NO
  [no lbl]  reference_centroid                        yes        yes        NO

  [lbl]     centroid_mesh, center_per_label=True      yes        NO         NO
  [lbl]     centroid_mesh, center_per_label=False     yes        yes        yes


---
## Center mode reference

### Mode descriptions

| Mode | Description |
|---|---|
| `center=None` | Use the mesh origin as the reference frame. No shift is applied; all tensors are fully additive. |
| `center=None` + manual pre-shift | Shift all vertices by the global centroid *before* calling `minkowski_tensors`. Equivalent to `centroid_mesh` on the combined mesh when the same shift is applied consistently; all tensors are additive. Similar to the case tested in MinK/MinT|
| `center='centroid_mesh'`, `center_per_label=True` | Each body is shifted to its own volume-weighted centre of mass ($W_0^{10}/W_0^{00}$) before computing. Scalars are additive; vectors and rank-2 tensors are **not** (different bodies are shifted by different amounts). |
| `center='centroid_mesh'`, `center_per_label=False` | All bodies are shifted by the single global centre of mass of the full mesh. Equivalent to the manual pre-shift above; all tensors are additive. |
| `center='reference_centroid'` | Direct port of the C++ `--reference_centroid` flag. Vertices are **not** shifted; instead, the per-tensor Minkowski centroid ($W_x^{10}/W_x^{00}$) is subtracted when building the rank-2 tensor integrand. Scalars and vectors are additive; rank-2 tensors are **not**. **Always computed per-label regardless of `center_per_label`** — no global-scope variant exists for this mode. |
| `center=<(3,) array>` | Shift all vertices by a user-supplied fixed point before computing. Additivity depends on whether the same point is used for both sub-meshes and the combined mesh. |

### Availability: C++ karambola vs pykarambola

| Feature | C++ karambola | pykarambola |
|---|---|---|
| Origin reference (`center=None`) | ✓ (default) | ✓ |
| `--reference_centroid` / `center='reference_centroid'` | ✓ (per-label when `--labels` used) | ✓ (always per-label; `center_per_label` ignored) |
| `center='centroid_mesh'` (volume-weighted CoM shift) | ✗ | ✓ |
| `center_per_label=True` / `'global'` | ✗ | ✓ (for `centroid_mesh` only) |
| Explicit array center | ✗ | ✓ |

> **Note:** `center_per_label=False` is only meaningful for `centroid_mesh`. The `reference_centroid` mode has no global-scope variant, matching the C++ implementation.